# Webhooks & FastAPI Triggers — Practical Walkthrough

In earlier sessions, agents were called from notebooks, scripts, or framework workflows. In this practical, the agent becomes an event-driven backend service: an external system sends an HTTP event, FastAPI validates it, and an agent runs in the background.

**What you will build and test**

1. Validate webhook JSON with Pydantic.
2. Sign and verify a simulated webhook payload.
3. Run a local FastAPI server.
4. Send a support-form event to a webhook endpoint.
5. Receive a quick `202 Accepted` response with a job ID.
6. Poll the job endpoint until the summarisation agent result is ready.


## Setup

The app runs in mock LLM mode by default. That means the complete webhook flow works without an API key. To use OpenAI, set `USE_MOCK_LLM=false` and provide `OPENAI_API_KEY` in `.env`.


In [7]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True)
import requests
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").exists():
    raise RuntimeError("Run this notebook from the webhooks_fastapi_triggers project root.")

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)
print("Mock LLM mode:", os.getenv("USE_MOCK_LLM", "true"))


Project root: e:\BIA\BIA_GenAI_May_26\webhooks_fastapi_triggers
Mock LLM mode: false


## Concept 1 — Validate the incoming event before the agent sees it

A webhook sender can send malformed data. The receiver should reject bad payloads before the agent logic runs. FastAPI uses Pydantic models to validate request bodies; here we use the same model directly in the notebook first.


In [8]:
from app.schemas import TextSubmittedEvent

payload = json.loads((PROJECT_ROOT / "data" / "form_event.json").read_text())
event = TextSubmittedEvent(**payload)

event


TextSubmittedEvent(event_id='evt_form_1001', event_type='support.form_submitted', source=<EventSource.FORM: 'form'>, text='I tried following the onboarding guide but the API key setup section is confusing. I do not know where to place the .env file or how to confirm that the key is loaded.', user_email='learner@example.com', metadata={'page': 'onboarding', 'customer_tier': 'trial', 'submitted_from': 'website_contact_form'}, received_at=datetime.datetime(2026, 7, 18, 4, 8, 15, 260057, tzinfo=datetime.timezone.utc))

Now intentionally break the payload to see validation in action. This is the same type of rejection FastAPI will return as a `422 Unprocessable Entity` response.


In [9]:
from pydantic import ValidationError

bad_payload = payload.copy()
bad_payload["text"] = "too short"

try:
    TextSubmittedEvent(**bad_payload)
except ValidationError as exc:
    print(exc)


1 validation error for TextSubmittedEvent
text
  String should have at least 10 characters [type=string_too_short, input_value='too short', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


## Concept 2 — Webhook signatures protect the receiver

Real webhook providers usually sign payloads. The receiver computes its own signature using the same secret and compares it with the signature header. This prevents random clients from pretending to be trusted event senders.


In [10]:
from app.security import build_signature

raw_body = (PROJECT_ROOT / "data" / "form_event.json").read_bytes()
secret = os.getenv("WEBHOOK_SECRET", "bia-demo-secret")

signature = build_signature(raw_body, secret)
signature


'sha256=08d7e1dd8f2ef24d2eda2be43b8c13fc6b3cc510c1169c1e3394ebdd0132a0e6'

## Concept 3 — The agent should produce a structured result

The agent returns a validated `AgentResult`, not unstructured prose. This makes downstream automation easier.


In [11]:
from app.agent import summarise_event

# COST NOTE: In default mock mode this costs nothing.
# In OpenAI mode, this short gpt-4o-mini call is typically far below one cent.
result = summarise_event(event)
result


AgentResult(summary='Customer is confused about API key setup in the onboarding guide.', category='support', priority='medium', recommended_action='Provide clear instructions on where to place the .env file and how to confirm the API key is loaded.')

## Concept 4 — Start the FastAPI server from the notebook

This cell starts Uvicorn in a subprocess. In a live classroom, the trainer may instead start the server from a separate terminal:

```bash
python -m uvicorn app.main:app --reload --port 8000
```


In [12]:
SERVER_URL = os.getenv("SERVER_URL", "http://127.0.0.1:8000")
PORT = SERVER_URL.rsplit(":", 1)[-1]

server_process = None

def server_is_ready(url: str) -> bool:
    """Return True when the local API server responds to /health."""
    try:
        response = requests.get(f"{url}/health", timeout=2)
        return response.status_code == 200
    except requests.RequestException:
        return False

if server_is_ready(SERVER_URL):
    print("Server already running:", SERVER_URL)
else:
    print("Starting server:", SERVER_URL)
    server_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app.main:app", "--port", PORT],
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for _ in range(20):
        if server_is_ready(SERVER_URL):
            print("Server is ready.")
            break
        time.sleep(0.5)
    else:
        raise RuntimeError("Server did not start. Check whether the port is already in use.")


Starting server: http://127.0.0.1:8000
Server is ready.


## Concept 5 — Send the webhook and get a fast acknowledgement

The endpoint should return quickly with a job ID. The final agent result is retrieved separately.


In [13]:
headers = {"Content-Type": "application/json", "X-BIA-Signature": signature}

response = requests.post(
    f"{SERVER_URL}/webhooks/text-submitted",
    data=raw_body,
    headers=headers,
    timeout=30,
)

print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))
response.raise_for_status()

job_id = response.json()["job_id"]
job_id


Status code: 202
{
  "job_id": "job_5cfc30cc1471",
  "status": "queued",
  "message": "Event accepted. Agent processing started in the background."
}


'job_5cfc30cc1471'

## Concept 6 — Poll the job endpoint until the agent result is ready

This is a simple classroom version of a production pattern. Real systems may push updates through WebSockets or store results in a database.


In [14]:
for attempt in range(10):
    status_response = requests.get(f"{SERVER_URL}/jobs/{job_id}", timeout=30)
    job_payload = status_response.json()
    print(f"Attempt {attempt + 1}: {job_payload['status']}")
    if job_payload["status"] in {"completed", "failed"}:
        print(json.dumps(job_payload, indent=2))
        break
    time.sleep(1)


Attempt 1: completed
{
  "job_id": "job_5cfc30cc1471",
  "status": "completed",
  "event_id": "evt_form_1001",
  "event_type": "support.form_submitted",
  "source": "form",
  "created_at": "2026-07-18T04:08:58.936091Z",
  "updated_at": "2026-07-18T04:09:03.717038Z",
  "result": {
    "summary": "Customer is confused about API key setup in onboarding guide.",
    "category": "support",
    "priority": "medium",
    "recommended_action": "Provide clear instructions on where to place the .env file and how to verify the API key is loaded."
  },
  "error": null
}


## Exercise 1 — Send the GitHub issue event

Use `data/github_issue_event.json` and send it to `/webhooks/github-issue`.

Hint:
- Read the file as bytes.
- Build a signature for that exact byte string.
- POST to `SERVER_URL + "/webhooks/github-issue"`.


In [15]:
# Try it:
github_raw_body = (PROJECT_ROOT / "data" / "github_issue_event.json").read_bytes()
github_signature = build_signature(github_raw_body, secret)

github_response = requests.post(
    f"{SERVER_URL}/webhooks/github-issue",
    data=github_raw_body,
    headers={"Content-Type": "application/json", "X-BIA-Signature": github_signature},
    timeout=30,
)

print("Status code:", github_response.status_code)
print(json.dumps(github_response.json(), indent=2))


Status code: 202
{
  "job_id": "job_67b62c44744d",
  "status": "queued",
  "message": "GitHub issue event accepted. Agent processing started in the background."
}


## Exercise 2 — Design a new email alert endpoint

The file `data/email_alert_event.json` already uses the generic `TextSubmittedEvent` shape. In the app, add a new endpoint:

```text
POST /webhooks/email-alert
```

It can reuse the same background processing function used by `/webhooks/text-submitted`.


## Closing cleanup

If this notebook started the server process, shut it down here.


In [16]:
if server_process is not None:
    server_process.terminate()
    try:
        server_process.wait(timeout=5)
        print("Server stopped.")
    except subprocess.TimeoutExpired:
        server_process.kill()
        print("Server killed after timeout.")
else:
    print("No notebook-owned server process to stop.")


Server stopped.


## What you built

You built a local event-driven agent backend:

```text
Webhook JSON → FastAPI endpoint → Pydantic validation → signature check → background task → summarisation agent → job status endpoint
```

Next, this trigger foundation can be extended so the agent streams its output token-by-token to a browser interface.
